In [1]:
import re
import torch
import unicodedata
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
import spacy
from transformers import pipeline
import difflib

c:\Users\super\anaconda3\envs\tcc2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download dos recursos necessários (executar uma vez)
def download_nltk_resources():
    try:
        nltk.download('punkt', quiet=True)
        nltk.download('stopwords', quiet=True)
        nltk.download('rslp', quiet=True)
    except:
        pass

In [3]:
class RobustTextProcessor:
    def __init__(self):
        """Inicializa o processador robusto de texto"""
        download_nltk_resources()
        
        # Carrega recursos NLTK
        try:
            self.stop_words = set(stopwords.words('portuguese'))
        except:
            self.stop_words = set()
        
        self.stemmer = RSLPStemmer()
        
        # Tenta carregar spaCy (opcional, para lematização mais avançada)
        try:
            self.nlp = spacy.load("pt_core_news_sm")
            self.spacy_available = True
        except:
            self.nlp = None
            self.spacy_available = False
            print("spaCy não encontrado. Usando apenas NLTK.")
        
        # Dicionário de correções comuns
        self.corrections_dict = {
            'ta': 'está',
            'tava': 'estava',
            'vc': 'você',
            'vcs': 'vocês',
            'pra': 'para',
            'pro': 'para o',
            'numa': 'em uma',
            'duma': 'de uma',
            'bom': 'bom',
            'mt': 'muito',
            'mto': 'muito',
            'msm': 'mesmo',
            'tbm': 'também',
            'td': 'tudo',
            'hj': 'hoje',
            'amanha': 'amanhã',
            'nao': 'não',
            'eh': 'é',
            'q': 'que',
            'oq': 'o que',
            'pq': 'porque',
            'blz': 'beleza',
            'vlw': 'valeu',
            'obg': 'obrigado',
            'brigado': 'obrigado'
        }
        
        # Padrões para separar palavras grudadas
        self.word_patterns = [
            (r'(\w+)(pelo)(\w+)', r'\1 \2 \3'),  # "travandopelo" -> "travando pelo"
            (r'(\w+)(para)(\w+)', r'\1 \2 \3'),  # "bomparao" -> "bom para o"
            (r'(\w+)(com)(\w+)', r'\1 \2 \3'),   # "produtocom" -> "produto com"
            (r'(\w+)(que)(\w+)', r'\1 \2 \3'),   # "dizque" -> "diz que"
            (r'(\w+)(mas)(\w+)', r'\1 \2 \3'),   # "bommas" -> "bom mas"
            (r'(\w+)(por)(\w+)', r'\1 \2 \3'),   # "feitopor" -> "feito por"
            (r'(\w+)(sem)(\w+)', r'\1 \2 \3'),   # "produtosem" -> "produto sem"
        ]
        
        # Classificador de sentimentos
        self.classifier = pipeline(
            "text-classification",
            model="neuralmind/bert-base-portuguese-cased",
            device=-1,
            return_all_scores=True
        )
    
    def fix_spacing_issues(self, text):
        """
        Corrige problemas de espaçamento entre palavras
        
        Args:
            text (str): Texto com possíveis problemas de espaçamento
        
        Returns:
            str: Texto com espaçamento corrigido
        """
        # Aplica padrões para separar palavras grudadas
        for pattern, replacement in self.word_patterns:
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        
        # Adiciona espaços antes de palavras comuns que podem estar grudadas
        common_words = ['pelo', 'para', 'com', 'que', 'mas', 'por', 'sem', 'da', 'do', 'na', 'no']
        for word in common_words:
            # Adiciona espaço antes da palavra se ela estiver grudada
            pattern = rf'([a-záàâãéêíóôõúç])({word})([a-záàâãéêíóôõúç])'
            text = re.sub(pattern, rf'\1 \2 \3', text, flags=re.IGNORECASE)
        
        return text
    
    def normalize_text(self, text):
        """
        Normaliza o texto (remove acentos, converte para minúsculo, etc.)
        
        Args:
            text (str): Texto a ser normalizado
        
        Returns:
            str: Texto normalizado
        """
        # Converte para minúsculo
        text = text.lower()
        
        # Remove caracteres especiais mas mantém espaços e letras com acento
        text = re.sub(r'[^\w\sáàâãéêíóôõúçñ]', ' ', text)
        
        # Remove espaços extras
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def expand_contractions(self, text):
        """
        Expande contrações e gírias comuns
        
        Args:
            text (str): Texto com possíveis contrações
        
        Returns:
            str: Texto com contrações expandidas
        """
        words = text.split()
        expanded_words = []
        
        for word in words:
            # Verifica se a palavra está no dicionário de correções
            if word in self.corrections_dict:
                expanded_words.append(self.corrections_dict[word])
            else:
                # Busca similaridade para correções automáticas
                similar = difflib.get_close_matches(word, self.corrections_dict.keys(), n=1, cutoff=0.8)
                if similar:
                    expanded_words.append(self.corrections_dict[similar[0]])
                else:
                    expanded_words.append(word)
        
        return ' '.join(expanded_words)
    
    def tokenize_and_lemmatize(self, text, remove_stopwords=True):
        """
        Tokeniza e lematiza o texto
        
        Args:
            text (str): Texto a ser processado
            remove_stopwords (bool): Se deve remover stopwords
        
        Returns:
            list: Lista de tokens/lemas
        """
        # Tokenização
        try:
            tokens = word_tokenize(text, language='portuguese')
        except:
            tokens = text.split()
        
        # Remove stopwords se solicitado
        if remove_stopwords and self.stop_words:
            tokens = [token for token in tokens if token.lower() not in self.stop_words]
        
        # Lematização usando spaCy se disponível
        if self.spacy_available and self.nlp:
            doc = self.nlp(' '.join(tokens))
            lemmas = [token.lemma_ for token in doc if not token.is_punct and len(token.text) > 1]
        else:
            # Usa stemming como alternativa
            lemmas = [self.stemmer.stem(token) for token in tokens if len(token) > 1]
        
        return [lemma for lemma in lemmas if lemma.strip()]
    
    def process_text(self, text, remove_stopwords=True):
        """
        Processo completo de limpeza e tokenização
        
        Args:
            text (str): Texto bruto
            remove_stopwords (bool): Se deve remover stopwords
        
        Returns:
            dict: Resultado do processamento
        """
        original = text
        
        # Passo 1: Corrigir espaçamento
        text = self.fix_spacing_issues(text)
        
        # Passo 2: Normalizar
        text = self.normalize_text(text)
        
        # Passo 3: Expandir contrações
        text = self.expand_contractions(text)
        
        # Passo 4: Tokenizar e lematizar
        tokens = self.tokenize_and_lemmatize(text, remove_stopwords)
        
        return {
            'original': original,
            'processed': text,
            'tokens': tokens,
            'reconstructed': ' '.join(tokens)
        }
    
    def classify_sentiment(self, text_or_tokens):
        """
        Classifica o sentimento do texto
        
        Args:
            text_or_tokens: Texto ou lista de tokens
        
        Returns:
            dict: Resultado da classificação
        """
        if isinstance(text_or_tokens, list):
            text = ' '.join(text_or_tokens)
        else:
            text = text_or_tokens
        
        results = self.classifier(text)
        
        # Mapeia para espectro
        pos_score = 0.0
        neg_score = 0.0
        
        for result in results[0]:
            label = result['label'].upper()
            score = result['score']
            
            if 'POSITIVE' in label or 'POS' in label:
                pos_score = score
            elif 'NEGATIVE' in label or 'NEG' in label:
                neg_score = score
        
        spectrum_score = pos_score - neg_score
        
        if spectrum_score > 0.3:
            sentiment = "Positivo"
        elif spectrum_score < -0.3:
            sentiment = "Negativo"
        else:
            sentiment = "Neutro"
        
        return {
            'spectrum_score': spectrum_score,
            'sentiment': sentiment,
            'positive_score': pos_score,
            'negative_score': neg_score
        }
    
    def analyze_complete(self, text):
        """
        Análise completa: processamento + classificação
        
        Args:
            text (str): Texto bruto
        
        Returns:
            dict: Resultado completo
        """
        # Processa o texto
        processing_result = self.process_text(text)
        
        # Classifica o sentimento
        sentiment_result = self.classify_sentiment(processing_result['tokens'])
        
        return {
            'processing': processing_result,
            'sentiment': sentiment_result
        }

In [4]:
# Exemplos de uso
def exemplos_uso():
    """Exemplos com diferentes tipos de texto problemático"""
    
    processor = RobustTextProcessor()
    
    textos_problematicos = [
        "pouco travandopelo valor ta bom",
        "produtomtbom recomendo pra vcs",
        "atendimentopessimo naocompromais",
        "entregaatrasada masqualidadeboa",
        "compreiehchegou rapidinhobom msm",
        "precoalto masvalea pena mt bom"
    ]
    
    print("=== Análise de Textos Problemáticos ===\n")
    
    for i, texto in enumerate(textos_problematicos, 1):
        print(f"Exemplo {i}:")
        print(f"Original: '{texto}'")
        
        resultado = processor.analyze_complete(texto)
        
        print(f"Processado: '{resultado['processing']['processed']}'")
        print(f"Tokens: {resultado['processing']['tokens']}")
        print(f"Sentimento: {resultado['sentiment']['sentiment']} (Score: {resultado['sentiment']['spectrum_score']:.3f})")
        print("-" * 60)

In [5]:
# Função simplificada para seu uso
def processar_e_classificar(texto):
    """
    Função simplificada para processar e classificar texto problemático
    
    Args:
        texto (str): Texto bruto com possíveis problemas
    
    Returns:
        dict: Resultado da análise
    """
    processor = RobustTextProcessor()
    return processor.analyze_complete(texto)

In [6]:
if __name__ == "__main__":
    exemplos_uso()
    
    # Exemplo específico
    print("\n=== Seu Exemplo Específico ===")
    resultado = processar_e_classificar("pouco travandopelo valor ta bom")
    print(f"Resultado: {resultado['sentiment']['sentiment']} ({resultado['sentiment']['spectrum_score']:.3f})")

spaCy não encontrado. Usando apenas NLTK.


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\super\anaconda3\envs\tcc2\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\super\.cache\huggingface\hub\models--neuralmind--bert-base-portuguese-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, 

=== Análise de Textos Problemáticos ===

Exemplo 1:
Original: 'pouco travandopelo valor ta bom'
Processado: 'pouco estava do pelo valor está bom'
Tokens: ['pouc', 'val', 'bom']
Sentimento: Neutro (Score: 0.000)
------------------------------------------------------------
Exemplo 2:
Original: 'produtomtbom recomendo pra vcs'
Processado: 'produtomtbom re com endo para vocês'
Tokens: ['produtomtbom', 're', 'end']
Sentimento: Neutro (Score: 0.000)
------------------------------------------------------------
Exemplo 3:
Original: 'atendimentopessimo naocompromais'
Processado: 'atendimentopessimo não com promais'
Tokens: ['atendimentopess', 'prom']
Sentimento: Neutro (Score: 0.000)
------------------------------------------------------------
Exemplo 4:
Original: 'entregaatrasada masqualidadeboa'
Processado: 'entregaatrasada masquali da deboa'
Tokens: ['entregaatras', 'masqual', 'debo']
Sentimento: Neutro (Score: 0.000)
------------------------------------------------------------
Exemplo 5:
Or

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


Resultado: Neutro (0.000)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
